In [1]:
"""
Complete Hyperparameter Tuning Pipeline for Sentiment Analysis
Supports: SVM, Random Forest, AdaBoost, LightGBM, XGBoost, LSTM, GRU, BiLSTM, TextCNN
With MLflow tracking and DVC versioning
"""

# ============================================================================
# File: config.py
# ============================================================================
from dataclasses import dataclass, field
from typing import List, Dict, Any
import os

@dataclass
class DataConfig:
    """Data configuration"""
    data_path: str
    text_column: str = "message"
    label_column: str = "sentiment"
    test_size: float = 0.2
    val_size: float = 0.1
    random_state: int = 42
    

@dataclass
class TuningConfig:
    """Hyperparameter tuning configuration"""
    n_trials: int = 50  # Number of trials for Optuna
    timeout: int = 3600  # Timeout in seconds
    n_jobs: int = -1  # Number of parallel jobs
    cv_folds: int = 5  # Cross-validation folds
    

@dataclass
class ModelSearchSpace:
    """Search space for each model"""
    
    # Traditional ML Models
    SVM: Dict[str, Any] = field(default_factory=lambda: {
        'C': ('float', 0.1, 100.0, 'log'),
        'kernel': ('categorical', ['linear', 'rbf', 'poly']),
        'gamma': ('categorical', ['scale', 'auto']),
        'class_weight': ('categorical', ['balanced', None])
    })
    
    RandomForest: Dict[str, Any] = field(default_factory=lambda: {
        'n_estimators': ('int', 50, 500),
        'max_depth': ('int', 5, 50),
        'min_samples_split': ('int', 2, 20),
        'min_samples_leaf': ('int', 1, 10),
        'max_features': ('categorical', ['sqrt', 'log2', None]),
        'class_weight': ('categorical', ['balanced', 'balanced_subsample', None])
    })
    
    AdaBoost: Dict[str, Any] = field(default_factory=lambda: {
        'n_estimators': ('int', 50, 500),
        'learning_rate': ('float', 0.01, 2.0, 'log'),
        'algorithm': ('categorical', ['SAMME', 'SAMME.R'])
    })
    
    LightGBM: Dict[str, Any] = field(default_factory=lambda: {
        'n_estimators': ('int', 50, 500),
        'learning_rate': ('float', 0.01, 0.3, 'log'),
        'max_depth': ('int', 3, 15),
        'num_leaves': ('int', 20, 150),
        'min_child_samples': ('int', 5, 100),
        'subsample': ('float', 0.5, 1.0),
        'colsample_bytree': ('float', 0.5, 1.0),
        'reg_alpha': ('float', 1e-8, 10.0, 'log'),
        'reg_lambda': ('float', 1e-8, 10.0, 'log')
    })
    
    XGBoost: Dict[str, Any] = field(default_factory=lambda: {
        'n_estimators': ('int', 50, 500),
        'learning_rate': ('float', 0.01, 0.3, 'log'),
        'max_depth': ('int', 3, 15),
        'min_child_weight': ('int', 1, 10),
        'subsample': ('float', 0.5, 1.0),
        'colsample_bytree': ('float', 0.5, 1.0),
        'gamma': ('float', 0, 5),
        'reg_alpha': ('float', 1e-8, 10.0, 'log'),
        'reg_lambda': ('float', 1e-8, 10.0, 'log')
    })
    
    # Deep Learning Models
    LSTM: Dict[str, Any] = field(default_factory=lambda: {
        'embedding_dim': ('categorical', [64, 128, 256]),
        'lstm_units': ('categorical', [64, 128, 256]),
        'lstm_layers': ('int', 1, 3),
        'dropout': ('float', 0.2, 0.6),
        'learning_rate': ('float', 1e-4, 1e-2, 'log'),
        'batch_size': ('categorical', [16, 32, 64]),
    })
    
    GRU: Dict[str, Any] = field(default_factory=lambda: {
        'embedding_dim': ('categorical', [64, 128, 256]),
        'gru_units': ('categorical', [64, 128, 256]),
        'gru_layers': ('int', 1, 3),
        'dropout': ('float', 0.2, 0.6),
        'learning_rate': ('float', 1e-4, 1e-2, 'log'),
        'batch_size': ('categorical', [16, 32, 64]),
    })
    
    BiLSTM: Dict[str, Any] = field(default_factory=lambda: {
        'embedding_dim': ('categorical', [64, 128, 256]),
        'lstm_units': ('categorical', [64, 128, 256]),
        'lstm_layers': ('int', 1, 3),
        'dropout': ('float', 0.2, 0.6),
        'learning_rate': ('float', 1e-4, 1e-2, 'log'),
        'batch_size': ('categorical', [16, 32, 64]),
    })
    
    TextCNN: Dict[str, Any] = field(default_factory=lambda: {
        'embedding_dim': ('categorical', [64, 128, 256]),
        'num_filters': ('categorical', [64, 128, 256]),
        'filter_sizes': ('categorical', [[2,3,4], [3,4,5], [2,3,4,5]]),
        'dropout': ('float', 0.2, 0.6),
        'learning_rate': ('float', 1e-4, 1e-2, 'log'),
        'batch_size': ('categorical', [16, 32, 64]),
    })


# ============================================================================
# File: preprocessing.py
# ============================================================================
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
import pickle
import json

class TextPreprocessor:
    """Unified text preprocessor for both traditional ML and deep learning"""
    
    def __init__(self, max_words=10000, max_len=100, use_tfidf=False, 
                 tfidf_max_features=5000):
        self.max_words = max_words
        self.max_len = max_len
        self.use_tfidf = use_tfidf
        self.tfidf_max_features = tfidf_max_features
        
        # For deep learning
        self.tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
        
        # For traditional ML
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=tfidf_max_features,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95
        )
        
        self.label_encoder = LabelEncoder()
        
    def clean_text(self, text):
        """Clean and normalize text"""
        text = str(text).lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'@\w+|#\w+', '', text)
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    def fit_transform(self, texts, labels, for_dl=True):
        """Fit and transform for either DL or traditional ML"""
        cleaned_texts = [self.clean_text(text) for text in texts]
        
        if for_dl:
            # Deep learning preprocessing
            self.tokenizer.fit_on_texts(cleaned_texts)
            sequences = self.tokenizer.texts_to_sequences(cleaned_texts)
            padded_sequences = pad_sequences(sequences, maxlen=self.max_len,
                                           padding='post', truncating='post')
            X = padded_sequences
        else:
            # Traditional ML preprocessing
            X = self.tfidf_vectorizer.fit_transform(cleaned_texts).toarray()
        
        encoded_labels = self.label_encoder.fit_transform(labels)
        return X, encoded_labels
    
    def transform(self, texts, labels=None, for_dl=True):
        """Transform texts and optionally labels"""
        cleaned_texts = [self.clean_text(text) for text in texts]
        
        if for_dl:
            sequences = self.tokenizer.texts_to_sequences(cleaned_texts)
            padded_sequences = pad_sequences(sequences, maxlen=self.max_len,
                                           padding='post', truncating='post')
            X = padded_sequences
        else:
            X = self.tfidf_vectorizer.transform(cleaned_texts).toarray()
        
        if labels is not None:
            encoded_labels = self.label_encoder.transform(labels)
            return X, encoded_labels
        return X
    
    def save(self, path):
        """Save preprocessor artifacts"""
        os.makedirs(path, exist_ok=True)
        
        with open(os.path.join(path, 'tokenizer.pkl'), 'wb') as f:
            pickle.dump(self.tokenizer, f)
        
        with open(os.path.join(path, 'tfidf_vectorizer.pkl'), 'wb') as f:
            pickle.dump(self.tfidf_vectorizer, f)
        
        with open(os.path.join(path, 'label_encoder.pkl'), 'wb') as f:
            pickle.dump(self.label_encoder, f)
        
        config = {
            'max_words': self.max_words,
            'max_len': self.max_len,
            'use_tfidf': self.use_tfidf,
            'tfidf_max_features': self.tfidf_max_features,
            'vocab_size': len(self.tokenizer.word_index) + 1,
            'num_classes': len(self.label_encoder.classes_),
            'classes': self.label_encoder.classes_.tolist()
        }
        with open(os.path.join(path, 'config.json'), 'w') as f:
            json.dump(config, f, indent=4)
    
    @classmethod
    def load(cls, path):
        """Load preprocessor artifacts"""
        with open(os.path.join(path, 'config.json'), 'r') as f:
            config = json.load(f)
        
        preprocessor = cls(
            max_words=config['max_words'],
            max_len=config['max_len'],
            use_tfidf=config.get('use_tfidf', False),
            tfidf_max_features=config.get('tfidf_max_features', 5000)
        )
        
        with open(os.path.join(path, 'tokenizer.pkl'), 'rb') as f:
            preprocessor.tokenizer = pickle.load(f)
        
        with open(os.path.join(path, 'tfidf_vectorizer.pkl'), 'rb') as f:
            preprocessor.tfidf_vectorizer = pickle.load(f)
        
        with open(os.path.join(path, 'label_encoder.pkl'), 'rb') as f:
            preprocessor.label_encoder = pickle.load(f)
        
        return preprocessor


def load_and_split_data(data_path, text_col, label_col,
                       test_size=0.2, val_size=0.1, random_state=42):
    """Load data and split into train/val/test sets"""
    df = pd.read_csv(data_path)
    
    print(f"Loaded dataset: {data_path}")
    print(f"Shape: {df.shape}")
    print(f"Label distribution:\n{df[label_col].value_counts()}")
    
    X = df[text_col].values
    y = df[label_col].values
    
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    val_ratio = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_ratio, random_state=random_state,
        stratify=y_temp
    )
    
    print(f"\nTrain size: {len(X_train)}")
    print(f"Val size: {len(X_val)}")
    print(f"Test size: {len(X_test)}")
    
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)


# ============================================================================
# File: models_ml.py
# ============================================================================
"""Traditional ML Models"""
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
import lightgbm as lgb
import xgboost as xgb

def get_ml_model(model_name, **params):
    """Factory function for traditional ML models"""
    
    models = {
        'SVM': SVC,
        'RandomForest': RandomForestClassifier,
        'AdaBoost': AdaBoostClassifier,
        'LightGBM': lgb.LGBMClassifier,
        'XGBoost': xgb.XGBClassifier
    }
    
    if model_name not in models:
        raise ValueError(f"Unknown model: {model_name}")
    
    # Add default parameters
    if model_name == 'SVM':
        params.setdefault('probability', True)
        params.setdefault('random_state', 42)
    elif model_name in ['RandomForest', 'AdaBoost']:
        params.setdefault('random_state', 42)
    elif model_name == 'LightGBM':
        params.setdefault('random_state', 42)
        params.setdefault('verbose', -1)
    elif model_name == 'XGBoost':
        params.setdefault('random_state', 42)
        params.setdefault('verbosity', 0)
        params.setdefault('use_label_encoder', False)
        params.setdefault('eval_metric', 'mlogloss')
    
    return models[model_name](**params)


# ============================================================================
# File: models_dl.py
# ============================================================================
"""Deep Learning Models"""
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

class LSTMModel:
    @staticmethod
    def build(vocab_size, embedding_dim, max_len, num_classes,
             lstm_units=128, lstm_layers=2, dropout=0.5):
        inputs = layers.Input(shape=(max_len,))
        x = layers.Embedding(vocab_size, embedding_dim, input_length=max_len)(inputs)
        
        for i in range(lstm_layers):
            return_sequences = (i < lstm_layers - 1)
            x = layers.LSTM(lstm_units, return_sequences=return_sequences)(x)
            x = layers.Dropout(dropout)(x)
        
        x = layers.Dense(64, activation='relu')(x)
        x = layers.Dropout(dropout)(x)
        outputs = layers.Dense(num_classes, activation='softmax')(x)
        
        return Model(inputs=inputs, outputs=outputs, name='LSTM')


class GRUModel:
    @staticmethod
    def build(vocab_size, embedding_dim, max_len, num_classes,
             gru_units=128, gru_layers=2, dropout=0.5):
        inputs = layers.Input(shape=(max_len,))
        x = layers.Embedding(vocab_size, embedding_dim, input_length=max_len)(inputs)
        
        for i in range(gru_layers):
            return_sequences = (i < gru_layers - 1)
            x = layers.GRU(gru_units, return_sequences=return_sequences)(x)
            x = layers.Dropout(dropout)(x)
        
        x = layers.Dense(64, activation='relu')(x)
        x = layers.Dropout(dropout)(x)
        outputs = layers.Dense(num_classes, activation='softmax')(x)
        
        return Model(inputs=inputs, outputs=outputs, name='GRU')


class BiLSTMModel:
    @staticmethod
    def build(vocab_size, embedding_dim, max_len, num_classes,
             lstm_units=128, lstm_layers=2, dropout=0.5):
        inputs = layers.Input(shape=(max_len,))
        x = layers.Embedding(vocab_size, embedding_dim, input_length=max_len)(inputs)
        
        for i in range(lstm_layers):
            return_sequences = (i < lstm_layers - 1)
            x = layers.Bidirectional(
                layers.LSTM(lstm_units, return_sequences=return_sequences)
            )(x)
            x = layers.Dropout(dropout)(x)
        
        x = layers.Dense(64, activation='relu')(x)
        x = layers.Dropout(dropout)(x)
        outputs = layers.Dense(num_classes, activation='softmax')(x)
        
        return Model(inputs=inputs, outputs=outputs, name='BiLSTM')


class TextCNNModel:
    @staticmethod
    def build(vocab_size, embedding_dim, max_len, num_classes,
             num_filters=128, filter_sizes=[3, 4, 5], dropout=0.5):
        inputs = layers.Input(shape=(max_len,))
        x = layers.Embedding(vocab_size, embedding_dim, input_length=max_len)(inputs)
        x = layers.Dropout(dropout)(x)
        
        conv_blocks = []
        for filter_size in filter_sizes:
            conv = layers.Conv1D(num_filters, filter_size, activation='relu')(x)
            conv = layers.GlobalMaxPooling1D()(conv)
            conv_blocks.append(conv)
        
        x = layers.Concatenate()(conv_blocks) if len(conv_blocks) > 1 else conv_blocks[0]
        x = layers.Dense(128, activation='relu')(x)
        x = layers.Dropout(dropout)(x)
        outputs = layers.Dense(num_classes, activation='softmax')(x)
        
        return Model(inputs=inputs, outputs=outputs, name='TextCNN')


def get_dl_model(model_name, **params):
    """Factory function for deep learning models"""
    models = {
        'LSTM': LSTMModel,
        'GRU': GRUModel,
        'BiLSTM': BiLSTMModel,
        'TextCNN': TextCNNModel
    }
    
    if model_name not in models:
        raise ValueError(f"Unknown model: {model_name}")
    
    return models[model_name].build(**params)


# ============================================================================
# File: hyperparameter_tuner.py
# ============================================================================
"""Hyperparameter tuning with Optuna"""
import optuna
from optuna.integration import MLflowCallback
import mlflow
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, f1_score
from tensorflow import keras
import tensorflow as tf

class HyperparameterTuner:
    """Unified hyperparameter tuner for ML and DL models"""
    
    def __init__(self, model_name, search_space, preprocessor, 
                 X_train, y_train, X_val, y_val, tuning_config):
        self.model_name = model_name
        self.search_space = search_space
        self.preprocessor = preprocessor
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.config = tuning_config
        self.is_dl_model = model_name in ['LSTM', 'GRU', 'BiLSTM', 'TextCNN']
        
    def suggest_params(self, trial):
        """Suggest parameters based on search space"""
        params = {}
        
        for param_name, param_config in self.search_space.items():
            param_type = param_config[0]
            
            if param_type == 'int':
                params[param_name] = trial.suggest_int(
                    param_name, param_config[1], param_config[2]
                )
            elif param_type == 'float':
                log = len(param_config) > 3 and param_config[3] == 'log'
                params[param_name] = trial.suggest_float(
                    param_name, param_config[1], param_config[2], log=log
                )
            elif param_type == 'categorical':
                params[param_name] = trial.suggest_categorical(
                    param_name, param_config[1]
                )
        
        return params
    
    def objective_ml(self, trial):
        """Objective function for traditional ML models"""
        params = self.suggest_params(trial)
        
        try:
            from models_ml import get_ml_model
            model = get_ml_model(self.model_name, **params)
            
            # Cross-validation
            scores = cross_val_score(
                model, self.X_train, self.y_train,
                cv=self.config.cv_folds,
                scoring='f1_weighted',
                n_jobs=self.config.n_jobs
            )
            
            return scores.mean()
            
        except Exception as e:
            print(f"Trial failed: {e}")
            return 0.0
    
    def objective_dl(self, trial):
        """Objective function for deep learning models"""
        params = self.suggest_params(trial)
        
        try:
            # Extract training params
            learning_rate = params.pop('learning_rate')
            batch_size = params.pop('batch_size')
            
            # Build model
            with open('artifacts/preprocessor/config.json', 'r') as f:
                import json
                prep_config = json.load(f)
            
            model_params = {
                'vocab_size': prep_config['vocab_size'],
                'embedding_dim': params.get('embedding_dim', 128),
                'max_len': prep_config['max_len'],
                'num_classes': prep_config['num_classes'],
                'dropout': params.get('dropout', 0.5)
            }
            
            if self.model_name == 'LSTM':
                model_params['lstm_units'] = params.get('lstm_units', 128)
                model_params['lstm_layers'] = params.get('lstm_layers', 2)
            elif self.model_name == 'GRU':
                model_params['gru_units'] = params.get('gru_units', 128)
                model_params['gru_layers'] = params.get('gru_layers', 2)
            elif self.model_name == 'BiLSTM':
                model_params['lstm_units'] = params.get('lstm_units', 128)
                model_params['lstm_layers'] = params.get('lstm_layers', 2)
            elif self.model_name == 'TextCNN':
                model_params['num_filters'] = params.get('num_filters', 128)
                model_params['filter_sizes'] = params.get('filter_sizes', [3,4,5])
            
            from models_dl import get_dl_model
            model = get_dl_model(self.model_name, **model_params)
            
            # Compile
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
            
            # Train
            early_stop = keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=3, restore_best_weights=True
            )
            
            history = model.fit(
                self.X_train, self.y_train,
                validation_data=(self.X_val, self.y_val),
                epochs=20,
                batch_size=batch_size,
                callbacks=[early_stop],
                verbose=0
            )
            
            # Evaluate
            val_loss, val_acc = model.evaluate(self.X_val, self.y_val, verbose=0)
            
            # Clean up
            keras.backend.clear_session()
            del model
            tf.compat.v1.reset_default_graph()
            
            return val_acc
            
        except Exception as e:
            print(f"Trial failed: {e}")
            keras.backend.clear_session()
            tf.compat.v1.reset_default_graph()
            return 0.0
    
    def tune(self):
        """Run hyperparameter tuning"""
        print(f"\n{'='*80}")
        print(f"Tuning {self.model_name}")
        print(f"{'='*80}\n")
        
        # Create study
        study = optuna.create_study(
            direction='maximize',
            study_name=f"{self.model_name}_tuning"
        )
        
        # Setup MLflow callback
        mlflc = MLflowCallback(
            tracking_uri=mlflow.get_tracking_uri(),
            metric_name='score'
        )
        
        # Run optimization
        objective = self.objective_dl if self.is_dl_model else self.objective_ml
        
        study.optimize(
            objective,
            n_trials=self.config.n_trials,
            timeout=self.config.timeout,
            callbacks=[mlflc],
            show_progress_bar=True
        )
        
        print(f"\nBest trial:")
        print(f"  Score: {study.best_trial.value:.4f}")
        print(f"  Params: {study.best_trial.params}")
        
        return study.best_params, study.best_value


# ============================================================================
# File: trainer.py
# ============================================================================
"""Model training with best hyperparameters"""
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                            classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns

class ModelTrainer:
    """Train and evaluate models with best hyperparameters"""
    
    def __init__(self, model_name, best_params, preprocessor):
        self.model_name = model_name
        self.best_params = best_params
        self.preprocessor = preprocessor
        self.model = None
        self.history = None
        self.is_dl_model = model_name in ['LSTM', 'GRU', 'BiLSTM', 'TextCNN']
    
    def train_ml(self, X_train, y_train, X_val, y_val):
        """Train traditional ML model"""
        from models_ml import get_ml_model
        
        self.model = get_ml_model(self.model_name, **self.best_params)
        self.model.fit(X_train, y_train)
        
        return self.model
    
    def train_dl(self, X_train, y_train, X_val, y_val):
        """Train deep learning model"""
        from models_dl import get_dl_model
        import json
        
        # Load config
        with open('artifacts/preprocessor/config.json', 'r') as f:
            prep_config = json.load(f)
        
        # Extract training params
        learning_rate = self.best_params.pop('learning_rate', 0.001)
        batch_size = self.best_params.pop('batch_size', 32)
        
        # Build model params
        model_params = {
            'vocab_size': prep_config['vocab_size'],
            'embedding_dim': self.best_params.get('embedding_dim', 128),
            'max_len': prep_config['max_len'],
            'num_classes': prep_config['num_classes'],
            'dropout': self.best_params.get('dropout', 0.5)
        }
        
        if self.model_name == 'LSTM':
            model_params['lstm_units'] = self.best_params.get('lstm_units', 128)
            model_params['lstm_layers'] = self.best_params.get('lstm_layers', 2)
        elif self.model_name == 'GRU':
            model_params['gru_units'] = self.best_params.get('gru_units', 128)
            model_params['gru_layers'] = self.best_params.get('gru_layers', 2)
        elif self.model_name == 'BiLSTM':
            model_params['lstm_units'] = self.best_params.get('lstm_units', 128)
            model_params['lstm_layers'] = self.best_params.get('lstm_layers', 2)
        elif self.model_name == 'TextCNN':
            model_params['num_filters'] = self.best_params.get('num_filters', 128)
            model_params['filter_sizes'] = self.best_params.get('filter_sizes', [3,4,5])
        
        self.model = get_dl_model(self.model_name, **model_params)
        
        # Compile
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Callbacks
        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=5, restore_best_weights=True
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=3
            )
        ]
        
        # Train
        self.history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1
        )
        
        return self.model
    
    def train(self, X_train, y_train, X_val, y_val):
        """Train model based on type"""
        if self.is_dl_model:
            return self.train_dl(X_train, y_train, X_val, y_val)
        else:
            return self.train_ml(X_train, y_train, X_val, y_val)
    
    def evaluate(self, X_test, y_test):
        """Evaluate model"""
        if self.is_dl_model:
            y_pred_probs = self.model.predict(X_test)
            y_pred = np.argmax(y_pred_probs, axis=1)
        else:
            y_pred = self.model.predict(X_test)
        
        accuracy = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred, average='weighted'
        )
        
        class_report = classification_report(
            y_test, y_pred,
            target_names=self.preprocessor.label_encoder.classes_,
            output_dict=True
        )
        
        cm = confusion_matrix(y_test, y_pred)
        
        metrics = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'confusion_matrix': cm,
            'classification_report': class_report
        }
        
        return metrics
    
    def plot_training_history(self, save_path):
        """Plot training history for DL models"""
        if not self.is_dl_model or self.history is None:
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        axes[0].plot(self.history.history['accuracy'], label='Train')
        axes[0].plot(self.history.history['val_accuracy'], label='Validation')
        axes[0].set_title('Model Accuracy')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Accuracy')
        axes[0].legend()
        axes[0].grid(True)
        
        axes[1].plot(self.history.history['loss'], label='Train')
        axes[1].plot(self.history.history['val_loss'], label='Validation')
        axes[1].set_title('Model Loss')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].legend()
        axes[1].grid(True)
        
        plt.tight_layout()
        plt.savefig(save_path)
        plt.close()
    
    def plot_confusion_matrix(self, cm, save_path):
        """Plot confusion matrix"""
        fig, ax = plt.subplots(figsize=(10, 8))
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=self.preprocessor.label_encoder.classes_,
                   yticklabels=self.preprocessor.label_encoder.classes_,
                   ax=ax)
        
        ax.set_title(f'{self.model_name} - Confusion Matrix')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        
        plt.tight_layout()
        plt.savefig(save_path)
        plt.close()
    
    def save_model(self, path):
        """Save model"""
        os.makedirs(path, exist_ok=True)
        
        if self.is_dl_model:
            self.model.save(os.path.join(path, 'model.h5'))
        else:
            import joblib
            joblib.dump(self.model, os.path.join(path, 'model.pkl'))


# ============================================================================
# File: pipeline.py
# ============================================================================
"""Complete MLOps pipeline with hyperparameter tuning"""
import mlflow
import mlflow.sklearn
import mlflow.keras
from datetime import datetime
import json

class MLOpsPipeline:
    """Complete pipeline with tuning, training, and tracking"""
    
    def __init__(self, experiment_name="sentiment-hyperparameter-tuning"):
        self.experiment_name = experiment_name
        mlflow.set_experiment(experiment_name)
    
    def run_pipeline(self, data_config, model_name, search_space, 
                    tuning_config, run_name=None):
        """Run complete pipeline for a model"""
        
        if run_name is None:
            run_name = f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        with mlflow.start_run(run_name=run_name) as run:
            print(f"\n{'='*80}")
            print(f"MLflow Run: {run_name}")
            print(f"Run ID: {run.info.run_id}")
            print(f"{'='*80}\n")
            
            # Log basic info
            mlflow.log_param("model_name", model_name)
            mlflow.log_param("data_path", data_config.data_path)
            mlflow.log_param("n_trials", tuning_config.n_trials)
            
            # Step 1: Load and split data
            print("[1/7] Loading and splitting data...")
            (X_train, y_train), (X_val, y_val), (X_test, y_test) = load_and_split_data(
                data_config.data_path,
                data_config.text_column,
                data_config.label_column,
                data_config.test_size,
                data_config.val_size,
                data_config.random_state
            )
            
            # Step 2: Preprocessing
            print("\n[2/7] Preprocessing data...")
            is_dl_model = model_name in ['LSTM', 'GRU', 'BiLSTM', 'TextCNN']
            
            preprocessor = TextPreprocessor(max_words=10000, max_len=100)
            
            X_train_proc, y_train_proc = preprocessor.fit_transform(
                X_train, y_train, for_dl=is_dl_model
            )
            X_val_proc, y_val_proc = preprocessor.transform(
                X_val, y_val, for_dl=is_dl_model
            )
            X_test_proc, y_test_proc = preprocessor.transform(
                X_test, y_test, for_dl=is_dl_model
            )
            
            # Save preprocessor
            prep_path = f"artifacts/preprocessor_{run.info.run_id}"
            preprocessor.save(prep_path)
            mlflow.log_artifacts(prep_path, artifact_path="preprocessor")
            
            print(f"Train samples: {len(X_train_proc)}")
            print(f"Val samples: {len(X_val_proc)}")
            print(f"Test samples: {len(X_test_proc)}")
            
            # Step 3: Hyperparameter tuning
            print(f"\n[3/7] Running hyperparameter tuning for {model_name}...")
            tuner = HyperparameterTuner(
                model_name, search_space, preprocessor,
                X_train_proc, y_train_proc,
                X_val_proc, y_val_proc,
                tuning_config
            )
            
            best_params, best_score = tuner.tune()
            
            # Log best hyperparameters
            mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})
            mlflow.log_metric("tuning_best_score", best_score)
            
            # Save tuning results
            tuning_results = {
                'best_params': best_params,
                'best_score': float(best_score),
                'model_name': model_name
            }
            
            tuning_path = f"artifacts/tuning_{run.info.run_id}"
            os.makedirs(tuning_path, exist_ok=True)
            with open(os.path.join(tuning_path, 'best_params.json'), 'w') as f:
                json.dump(tuning_results, f, indent=4)
            mlflow.log_artifacts(tuning_path, artifact_path="tuning_results")
            
            # Step 4: Train with best parameters
            print(f"\n[4/7] Training {model_name} with best hyperparameters...")
            trainer = ModelTrainer(model_name, best_params.copy(), preprocessor)
            trainer.train(X_train_proc, y_train_proc, X_val_proc, y_val_proc)
            
            # Step 5: Evaluate
            print("\n[5/7] Evaluating model...")
            metrics = trainer.evaluate(X_test_proc, y_test_proc)
            
            # Log metrics
            mlflow.log_metric("test_accuracy", metrics['accuracy'])
            mlflow.log_metric("test_precision", metrics['precision'])
            mlflow.log_metric("test_recall", metrics['recall'])
            mlflow.log_metric("test_f1_score", metrics['f1_score'])
            
            # Log per-class metrics
            for class_name, class_metrics in metrics['classification_report'].items():
                if isinstance(class_metrics, dict):
                    for metric_name, value in class_metrics.items():
                        if metric_name != 'support':
                            mlflow.log_metric(
                                f"{class_name}_{metric_name}", value
                            )
            
            print("\n" + "="*80)
            print("TEST RESULTS")
            print("="*80)
            print(f"Accuracy:  {metrics['accuracy']:.4f}")
            print(f"Precision: {metrics['precision']:.4f}")
            print(f"Recall:    {metrics['recall']:.4f}")
            print(f"F1 Score:  {metrics['f1_score']:.4f}")
            
            # Step 6: Save artifacts
            print("\n[6/7] Saving artifacts...")
            plot_path = f"artifacts/plots_{run.info.run_id}"
            os.makedirs(plot_path, exist_ok=True)
            
            if is_dl_model:
                trainer.plot_training_history(
                    os.path.join(plot_path, "training_history.png")
                )
            
            trainer.plot_confusion_matrix(
                metrics['confusion_matrix'],
                os.path.join(plot_path, "confusion_matrix.png")
            )
            
            mlflow.log_artifacts(plot_path, artifact_path="plots")
            
            # Save model
            model_path = f"models/{model_name}_{run.info.run_id}"
            trainer.save_model(model_path)
            
            if is_dl_model:
                mlflow.keras.log_model(trainer.model, "model")
            else:
                mlflow.sklearn.log_model(trainer.model, "model")
            
            # Step 7: Save metrics for DVC
            print("\n[7/7] Saving DVC metrics...")
            metrics_output = {
                'accuracy': float(metrics['accuracy']),
                'precision': float(metrics['precision']),
                'recall': float(metrics['recall']),
                'f1_score': float(metrics['f1_score']),
                'best_params': best_params,
                'tuning_score': float(best_score)
            }
            
            os.makedirs('metrics', exist_ok=True)
            with open(f'metrics/{model_name}_metrics.json', 'w') as f:
                json.dump(metrics_output, f, indent=4)
            
            print(f"\n{'='*80}")
            print(f"Pipeline completed successfully!")
            print(f"Run ID: {run.info.run_id}")
            print(f"{'='*80}\n")
            
            return run.info.run_id, metrics, best_params


# ============================================================================
# File: main.py
# ============================================================================
"""
Main execution script for hyperparameter tuning pipeline
"""

def main():
    """Main execution function"""
    
    # Configure datasets
    datasets = [
        {
            'name': 'dataset1',
            'path': 'data/raw/sentiment_dataset1.csv',
            'text_col': 'message',
            'label_col': 'sentiment'
        },
        # Add more datasets as needed
    ]
    
    # Configure models to tune
    models_to_tune = [
        'SVM',
        'RandomForest', 
        'AdaBoost',
        'LightGBM',
        'XGBoost',
        'LSTM',
        'GRU',
        'BiLSTM',
        'TextCNN'
    ]
    
    # Tuning configuration
    tuning_config = TuningConfig(
        n_trials=30,  # Number of trials per model
        timeout=3600,  # 1 hour timeout
        n_jobs=-1,
        cv_folds=5
    )
    
    # Initialize search spaces
    search_spaces = ModelSearchSpace()
    
    # Initialize pipeline
    pipeline = MLOpsPipeline(experiment_name="sentiment-hyperparameter-tuning")
    
    # Store all results
    all_results = {}
    
    for dataset in datasets:
        print(f"\n{'#'*80}")
        print(f"Processing Dataset: {dataset['name']}")
        print(f"{'#'*80}\n")
        
        data_config = DataConfig(
            data_path=dataset['path'],
            text_column=dataset['text_col'],
            label_column=dataset['label_col']
        )
        
        dataset_results = {}
        
        for model_name in models_to_tune:
            print(f"\n{'*'*80}")
            print(f"Tuning and Training: {model_name}")
            print(f"{'*'*80}\n")
            
            # Get search space for model
            search_space = getattr(search_spaces, model_name)
            
            try:
                run_id, metrics, best_params = pipeline.run_pipeline(
                    data_config,
                    model_name,
                    search_space,
                    tuning_config,
                    run_name=f"{dataset['name']}_{model_name}"
                )
                
                dataset_results[model_name] = {
                    'run_id': run_id,
                    'metrics': metrics,
                    'best_params': best_params
                }
                
                print(f"\n✓ {model_name} completed successfully!")
                print(f"  Best F1 Score: {metrics['f1_score']:.4f}")
                
            except Exception as e:
                print(f"\n✗ {model_name} failed: {str(e)}")
                continue
        
        all_results[dataset['name']] = dataset_results
    
    # Print final summary
    print("\n" + "="*80)
    print("EXPERIMENT SUMMARY")
    print("="*80)
    
    for dataset_name, dataset_results in all_results.items():
        print(f"\nDataset: {dataset_name}")
        print("-" * 40)
        
        # Sort by F1 score
        sorted_results = sorted(
            dataset_results.items(),
            key=lambda x: x[1]['metrics']['f1_score'],
            reverse=True
        )
        
        for rank, (model_name, result) in enumerate(sorted_results, 1):
            print(f"\n{rank}. {model_name}:")
            print(f"   Run ID: {result['run_id']}")
            print(f"   Accuracy: {result['metrics']['accuracy']:.4f}")
            print(f"   F1 Score: {result['metrics']['f1_score']:.4f}")
            print(f"   Best Params: {result['best_params']}")
    
    print("\n" + "="*80)
    print("All experiments completed!")
    print("View results in MLflow UI: http://localhost:5000")
    print("="*80 + "\n")
    
    # Save comparison results
    comparison_results = {}
    for dataset_name, dataset_results in all_results.items():
        comparison_results[dataset_name] = {
            model: {
                'accuracy': float(res['metrics']['accuracy']),
                'f1_score': float(res['metrics']['f1_score']),
                'best_params': res['best_params']
            }
            for model, res in dataset_results.items()
        }
    
    os.makedirs('metrics', exist_ok=True)
    with open('metrics/all_models_comparison.json', 'w') as f:
        json.dump(comparison_results, f, indent=4)


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'tensorflow'